# Schema-Retention Ablation: OCSVM Only, `dedup_on`

هذا الدفتر الصغير مخصص لإغلاق ملاحظة منهجية واحدة في الورقة:

> ماذا يحدث إذا أبقينا الأعمدة السبعة الموجودة في ملفات الهجوم فقط بدلاً من حذفها؟

التجربة هنا ليست إعادة تشغيل كاملة، بل **stress test** فقط:

- النموذج: `OCSVM` فقط.
- الإعداد: `dedup_on` فقط.
- لا يوجد LOF.
- لا يوجد Isolation Forest.
- لا يوجد repeated seeds.
- لا يوجد Bootstrap.
- لا يوجد attack-train audit.
- لا يغيّر نتائج الورقة الأساسية، بل يضيف ablation diagnostic.

الأعمدة السبعة:

```text
Duration, Srate, Drate, Magnitue, Radius, Covariance, Weight
```

منهجياً:
- يتم استخدام **39-feature profiling-only schema** نفسها لحساب row hashes وdedup/split حتى لا نخلط أثر الأعمدة السبعة مع أثر تغيير التقسيم.
- يتم تدريب OCSVM على **46 feature** = 39 الأصلية + 7 retained attack-side-only columns.
- الأعمدة السبعة غير الموجودة في profiling تُملأ بـ `0.0` في profiling، وتُترك بقيمها الحقيقية في attack-test.
- هذا ليس تصميماً مقترحاً للنشر، بل اختبار يقيس مدى خطورة schema retention.


In [1]:
# وظيفة الخلية: ربط Google Drive حتى يقرأ الدفتر الداتا الأصلية ويحفظ مخرجات ablation.

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# وظيفة الخلية: استيراد المكتبات وضبط الإعدادات العامة لتجربة OCSVM retained-schema فقط.

import os
import re
import gc
import glob
import json
import math
import time
import zipfile
import shutil
import hashlib
import subprocess
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM
from sklearn.metrics import (
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

# إعدادات ثابتة لمطابقة البروتوكول الأصلي.
RANDOM_STATE = 42
TRAIN_RATIO = 0.60
VALID_RATIO = 0.20
TEST_RATIO = 0.20
SENTINEL = -999999.0

# الأعمدة السبعة موضع الاختبار. ملاحظة: Magnitue هو تهجئة dataset-native كما في CICIoMT2024.
EXTRA_ATTACK_SIDE_COLS = [
    "Duration",
    "Srate",
    "Drate",
    "Magnitue",
    "Radius",
    "Covariance",
    "Weight",
]

META_COLS = ["__source_file", "__source_group", "__aux_label"]

# ضع مسار WiFI_and_MQTT يدوياً إذا لم يجده البحث التلقائي.
# يجب أن يكون المسار إلى مجلد WiFI_and_MQTT نفسه.
# مثال:
# DATASET_ROOT_MANUAL = "/content/drive/MyDrive/IoMT_Project/DataSet/CICIoMT2024/WiFI_and_MQTT"
DATASET_ROOT_MANUAL = ""

# اختيارياً: ضع مسار نتائج التشغيل النهائي السابقة حتى يحمل صف baseline تلقائياً للمقارنة.
# إذا تركته فارغاً، سيحاول الدفتر البحث عن publication_full*.zip أو result folder.
MANUAL_BASELINE_RESULTS_ROOT = ""
MANUAL_BASELINE_RESULTS_ZIP = ""

# لا نقرأ CSV مباشرة من Drive أثناء التحليل. ننسخ profiling وattack-test إلى /content أولاً.
STAGE_DATA_TO_LOCAL = True
LOCAL_STAGING_ROOT = "/content/schema_retention_staged_data"
LOCAL_DATASET_ROOT = os.path.join(LOCAL_STAGING_ROOT, "WiFI_and_MQTT")

# مجلد النتائج.
RUN_TAG = datetime.now().strftime("schema_retention_ablation_%Y%m%d_%H%M%S")
LOCAL_OUTPUT_ROOT = os.path.join("/content", RUN_TAG)

# سيتم نسخ النتائج النهائية إلى Google Drive في النهاية.
DRIVE_ROOT = "/content/drive/MyDrive"
DRIVE_OUTPUT_PARENT = os.path.join(DRIVE_ROOT, "schema_retention_ablation_results")

print("RUN_TAG:", RUN_TAG)
print("EXTRA_ATTACK_SIDE_COLS:", EXTRA_ATTACK_SIDE_COLS)


RUN_TAG: schema_retention_ablation_20260507_005607
EXTRA_ATTACK_SIDE_COLS: ['Duration', 'Srate', 'Drate', 'Magnitue', 'Radius', 'Covariance', 'Weight']


In [3]:
# وظيفة الخلية: إنشاء مجلدات الإخراج وحفظ إعدادات التجربة قبل التشغيل.

CONFIG_DIR = os.path.join(LOCAL_OUTPUT_ROOT, "config")
METRICS_DIR = os.path.join(LOCAL_OUTPUT_ROOT, "metrics")
PUBLICATION_DIR = os.path.join(LOCAL_OUTPUT_ROOT, "publication_checks")
AUDITS_DIR = os.path.join(LOCAL_OUTPUT_ROOT, "audits")
SUMMARIES_DIR = os.path.join(LOCAL_OUTPUT_ROOT, "summaries")

for d in [CONFIG_DIR, METRICS_DIR, PUBLICATION_DIR, AUDITS_DIR, SUMMARIES_DIR]:
    os.makedirs(d, exist_ok=True)

def json_default(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, (np.bool_,)):
        return bool(o)
    return str(o)

def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=json_default)

RUN_CONFIG = {
    "purpose": "schema_retention_ablation_ocsvm_only",
    "random_state": RANDOM_STATE,
    "train_ratio": TRAIN_RATIO,
    "valid_ratio": VALID_RATIO,
    "test_ratio": TEST_RATIO,
    "extra_attack_side_cols": EXTRA_ATTACK_SIDE_COLS,
    "identity_hash_schema": "profiling_only_39_features",
    "model_schema": "profiling_only_39_features_plus_7_attack_side_only_columns",
    "extra_column_policy": "extra columns absent from profiling are zero-filled; attack-test values are retained",
    "model": "OneClassSVM(kernel='rbf', gamma='scale', nu=0.05)",
    "threshold_policy": "95th percentile of normal_validation anomaly scores",
}
save_json(RUN_CONFIG, os.path.join(CONFIG_DIR, "schema_retention_ablation_config.json"))

print("Output root:", LOCAL_OUTPUT_ROOT)


Output root: /content/schema_retention_ablation_20260507_005607


In [4]:
# وظيفة الخلية: تعريف دوال البحث عن مسار WiFI_and_MQTT في Google Drive دون الاعتماد على مسار ثابت.

def is_wifi_mqtt_root(path):
    path = Path(path)
    return (
        (path / "profiling" / "CSV").is_dir()
        and (path / "attacks" / "CSV" / "test").is_dir()
    )

def depth_from_root(path, root):
    path = os.path.abspath(path)
    root = os.path.abspath(root)
    rel = os.path.relpath(path, root)
    if rel == ".":
        return 0
    return rel.count(os.sep) + 1

def find_wifi_mqtt_roots(base_dir, max_depth=7, max_hits=20):
    hits = []
    skip_names = {
        ".Trash",
        ".shortcut-targets-by-id",
        ".file-revisions-by-id",
        "__MACOSX",
        ".ipynb_checkpoints",
    }
    for root, dirs, files in os.walk(base_dir):
        dirs[:] = [d for d in dirs if d not in skip_names]
        if depth_from_root(root, base_dir) > max_depth:
            dirs[:] = []
            continue
        name = os.path.basename(root)
        if name in ("WiFI_and_MQTT", "WiFi_and_MQTT", "WIFI_and_MQTT") and is_wifi_mqtt_root(root):
            hits.append(root)
            if len(hits) >= max_hits:
                break
    return hits

def resolve_dataset_root():
    if DATASET_ROOT_MANUAL:
        if not is_wifi_mqtt_root(DATASET_ROOT_MANUAL):
            raise FileNotFoundError(f"DATASET_ROOT_MANUAL is not a valid WiFI_and_MQTT root: {DATASET_ROOT_MANUAL}")
        return DATASET_ROOT_MANUAL

    candidates = [
        os.path.join(DRIVE_ROOT, "IoMT_Project", "DataSet", "CICIoMT2024", "WiFI_and_MQTT"),
        os.path.join(DRIVE_ROOT, "DataSet", "CICIoMT2024", "WiFI_and_MQTT"),
        os.path.join(DRIVE_ROOT, "datasets", "CICIoMT2024", "WiFI_and_MQTT"),
        os.path.join(DRIVE_ROOT, "CICIoMT2024", "WiFI_and_MQTT"),
    ]
    for p in candidates:
        if is_wifi_mqtt_root(p):
            return p

    print("Searching Drive for WiFI_and_MQTT...")
    roots = find_wifi_mqtt_roots(DRIVE_ROOT)
    if roots:
        print("Found candidates:")
        for i, p in enumerate(roots, 1):
            print(f"{i}. {p}")
        return roots[0]

    raise FileNotFoundError(
        "Could not find WiFI_and_MQTT automatically. "
        "Set DATASET_ROOT_MANUAL in the configuration cell."
    )

DATASET_ROOT = resolve_dataset_root()
print("Selected DATASET_ROOT:", DATASET_ROOT)


Selected DATASET_ROOT: /content/drive/MyDrive/IoMT_Project/DataSet/CICIoMT2024/WiFI_and_MQTT


In [5]:
# وظيفة الخلية: نسخ profiling و attack-test CSV إلى /content لتجنب انقطاع Google Drive أثناء القراءة.

def copy_tree_with_rsync_or_python(src_dir, dst_dir):
    os.makedirs(dst_dir, exist_ok=True)
    if not os.path.isdir(src_dir):
        raise FileNotFoundError(src_dir)

    cmd = ["rsync", "-a", "--ignore-existing", src_dir.rstrip("/") + "/", dst_dir.rstrip("/") + "/"]
    try:
        result = subprocess.run(cmd, check=False, capture_output=True, text=True)
        if result.returncode == 0:
            return {"method": "rsync", "src": src_dir, "dst": dst_dir}
        print("rsync failed; falling back to shutil.copy2. stderr:", result.stderr[:500])
    except Exception as e:
        print("rsync unavailable; falling back to shutil.copy2:", type(e).__name__, e)

    for fp in glob.glob(os.path.join(src_dir, "*.csv")):
        target = os.path.join(dst_dir, os.path.basename(fp))
        if not os.path.exists(target):
            shutil.copy2(fp, target)
    return {"method": "python_copy", "src": src_dir, "dst": dst_dir}

def stage_dataset_to_local(dataset_root):
    if not STAGE_DATA_TO_LOCAL:
        return dataset_root, []

    src_profile = os.path.join(dataset_root, "profiling", "CSV")
    src_attack_test = os.path.join(dataset_root, "attacks", "CSV", "test")

    dst_profile = os.path.join(LOCAL_DATASET_ROOT, "profiling", "CSV")
    dst_attack_test = os.path.join(LOCAL_DATASET_ROOT, "attacks", "CSV", "test")

    print("Staging profiling CSVs...")
    rec1 = copy_tree_with_rsync_or_python(src_profile, dst_profile)

    print("Staging attack-test CSVs...")
    rec2 = copy_tree_with_rsync_or_python(src_attack_test, dst_attack_test)

    return LOCAL_DATASET_ROOT, [rec1, rec2]

ACTIVE_DATASET_ROOT, STAGING_RECORDS = stage_dataset_to_local(DATASET_ROOT)
print("ACTIVE_DATASET_ROOT:", ACTIVE_DATASET_ROOT)
print(json.dumps(STAGING_RECORDS, ensure_ascii=False, indent=2))


Staging profiling CSVs...
Staging attack-test CSVs...
ACTIVE_DATASET_ROOT: /content/schema_retention_staged_data/WiFI_and_MQTT
[
  {
    "method": "rsync",
    "src": "/content/drive/MyDrive/IoMT_Project/DataSet/CICIoMT2024/WiFI_and_MQTT/profiling/CSV",
    "dst": "/content/schema_retention_staged_data/WiFI_and_MQTT/profiling/CSV"
  },
  {
    "method": "rsync",
    "src": "/content/drive/MyDrive/IoMT_Project/DataSet/CICIoMT2024/WiFI_and_MQTT/attacks/CSV/test",
    "dst": "/content/schema_retention_staged_data/WiFI_and_MQTT/attacks/CSV/test"
  }
]


In [6]:
# وظيفة الخلية: تحميل CSV الخاصة بـ profiling و attack-test فقط مع الاحتفاظ باسم الملف والمجموعة.

def downcast_numeric(df):
    out = df.copy()
    for c in out.select_dtypes(include=["float64"]).columns:
        out[c] = pd.to_numeric(out[c], downcast="float")
    for c in out.select_dtypes(include=["int64"]).columns:
        out[c] = pd.to_numeric(out[c], downcast="integer")
    return out

def infer_aux_label(filename, source_group):
    if source_group == "profiling":
        return "normal"
    return "normal" if "benign" in filename.lower() else "attack"

def load_csv_group(csv_dir, source_group):
    files = sorted(glob.glob(os.path.join(csv_dir, "*.csv")))
    if not files:
        raise FileNotFoundError(f"No CSV files in {csv_dir}")
    frames = []
    audit = []
    for i, fp in enumerate(files, 1):
        df = pd.read_csv(fp)
        df = downcast_numeric(df)
        df["__source_file"] = os.path.basename(fp)
        df["__source_group"] = source_group
        df["__aux_label"] = infer_aux_label(os.path.basename(fp), source_group)
        frames.append(df)
        audit.append({"file": os.path.basename(fp), "rows": int(len(df)), "columns": int(len(df.columns))})
        if i % 10 == 0 or i == len(files):
            print(f"Loaded {source_group}: {i}/{len(files)} files")
    return pd.concat(frames, ignore_index=True), audit

profiling_dir = os.path.join(ACTIVE_DATASET_ROOT, "profiling", "CSV")
attack_test_dir = os.path.join(ACTIVE_DATASET_ROOT, "attacks", "CSV", "test")

profiling_raw, profiling_file_audit = load_csv_group(profiling_dir, "profiling")
attack_test_raw, attack_test_file_audit = load_csv_group(attack_test_dir, "attack_test")

print("Profiling rows:", len(profiling_raw), "files:", len(profiling_file_audit))
print("Attack-test rows:", len(attack_test_raw), "files:", len(attack_test_file_audit))

pd.DataFrame(profiling_file_audit).to_csv(os.path.join(AUDITS_DIR, "profiling_file_audit.csv"), index=False)
pd.DataFrame(attack_test_file_audit).to_csv(os.path.join(AUDITS_DIR, "attack_test_file_audit.csv"), index=False)


Loaded profiling: 10/42 files
Loaded profiling: 20/42 files
Loaded profiling: 30/42 files
Loaded profiling: 40/42 files
Loaded profiling: 42/42 files
Loaded attack_test: 10/21 files
Loaded attack_test: 20/21 files
Loaded attack_test: 21/21 files
Profiling rows: 389956 files: 42
Attack-test rows: 1614182 files: 21


In [7]:
# وظيفة الخلية: بناء schema الأصلي من profiling فقط ثم بناء retained schema بإضافة الأعمدة السبعة.

def profiling_feature_schema(profiling_df):
    numeric_cols = profiling_df.select_dtypes(include=[np.number]).columns.tolist()
    schema = [c for c in numeric_cols if not c.startswith("__") and c not in ["__binary_label"]]
    return sorted(schema)

BASE_SCHEMA_39 = profiling_feature_schema(profiling_raw)
RETAINED_SCHEMA_46 = BASE_SCHEMA_39 + [c for c in EXTRA_ATTACK_SIDE_COLS if c not in BASE_SCHEMA_39]

schema_info = {
    "base_schema_39_count": len(BASE_SCHEMA_39),
    "base_schema_39": BASE_SCHEMA_39,
    "extra_attack_side_cols": EXTRA_ATTACK_SIDE_COLS,
    "retained_schema_count": len(RETAINED_SCHEMA_46),
    "retained_schema_46": RETAINED_SCHEMA_46,
}

save_json(schema_info, os.path.join(CONFIG_DIR, "feature_schema_retained_46.json"))

print("BASE_SCHEMA_39:", len(BASE_SCHEMA_39))
print(BASE_SCHEMA_39)
print("\\nRETAINED_SCHEMA_46:", len(RETAINED_SCHEMA_46))
print(RETAINED_SCHEMA_46)


BASE_SCHEMA_39: 39
['ARP', 'AVG', 'DHCP', 'DNS', 'HTTP', 'HTTPS', 'Header_Length', 'IAT', 'ICMP', 'IGMP', 'IPv', 'IRC', 'LLC', 'Max', 'Min', 'Number', 'Protocol Type', 'Rate', 'SMTP', 'SSH', 'Std', 'TCP', 'Telnet', 'Time_To_Live', 'Tot size', 'Tot sum', 'UDP', 'Variance', 'ack_count', 'ack_flag_number', 'cwr_flag_number', 'ece_flag_number', 'fin_count', 'fin_flag_number', 'psh_flag_number', 'rst_count', 'rst_flag_number', 'syn_count', 'syn_flag_number']
\nRETAINED_SCHEMA_46: 46
['ARP', 'AVG', 'DHCP', 'DNS', 'HTTP', 'HTTPS', 'Header_Length', 'IAT', 'ICMP', 'IGMP', 'IPv', 'IRC', 'LLC', 'Max', 'Min', 'Number', 'Protocol Type', 'Rate', 'SMTP', 'SSH', 'Std', 'TCP', 'Telnet', 'Time_To_Live', 'Tot size', 'Tot sum', 'UDP', 'Variance', 'ack_count', 'ack_flag_number', 'cwr_flag_number', 'ece_flag_number', 'fin_count', 'fin_flag_number', 'psh_flag_number', 'rst_count', 'rst_flag_number', 'syn_count', 'syn_flag_number', 'Duration', 'Srate', 'Drate', 'Magnitue', 'Radius', 'Covariance', 'Weight']


In [8]:
# وظيفة الخلية: محاذاة الجداول إلى retained schema مع تعبئة الأعمدة السبعة المفقودة في profiling بقيمة صفر.

def apply_retained_schema(df, base_schema, retained_schema, extra_cols):
    out = df.copy()

    # الأعمدة الأساسية المفقودة تُترك NaN ثم تُعالج بـ median imputer مثل Time_To_Live في attack-test.
    for c in base_schema:
        if c not in out.columns:
            out[c] = np.nan

    # الأعمدة السبعة retained: إذا كانت غير موجودة في profiling أو أي ملف، تُملأ بصفر حتى نختبر أثر schema retention.
    for c in extra_cols:
        if c not in out.columns:
            out[c] = 0.0

    for c in META_COLS:
        if c not in out.columns:
            out[c] = np.nan

    aligned = out.reindex(columns=retained_schema + META_COLS, fill_value=np.nan)

    for c in retained_schema:
        aligned[c] = pd.to_numeric(aligned[c], errors="coerce")

    # سياسة stress test: extra columns missing/NaN are zero-filled, not median-imputed.
    for c in extra_cols:
        aligned[c] = aligned[c].replace([np.inf, -np.inf], np.nan).fillna(0.0)

    return aligned

profiling_aligned = apply_retained_schema(profiling_raw, BASE_SCHEMA_39, RETAINED_SCHEMA_46, EXTRA_ATTACK_SIDE_COLS)
attack_test_aligned = apply_retained_schema(attack_test_raw, BASE_SCHEMA_39, RETAINED_SCHEMA_46, EXTRA_ATTACK_SIDE_COLS)

schema_mismatch = {
    "attack_test_missing_base_cols": [c for c in BASE_SCHEMA_39 if c not in attack_test_raw.columns],
    "attack_test_extra_cols_retained": [c for c in EXTRA_ATTACK_SIDE_COLS if c in attack_test_raw.columns],
    "profiling_extra_cols_zero_filled": [c for c in EXTRA_ATTACK_SIDE_COLS if c not in profiling_raw.columns],
}
save_json(schema_mismatch, os.path.join(CONFIG_DIR, "schema_retention_policy.json"))

print(json.dumps(schema_mismatch, ensure_ascii=False, indent=2))
print("Profiling aligned:", profiling_aligned.shape)
print("Attack-test aligned:", attack_test_aligned.shape)


{
  "attack_test_missing_base_cols": [
    "Time_To_Live"
  ],
  "attack_test_extra_cols_retained": [
    "Duration",
    "Srate",
    "Drate",
    "Magnitue",
    "Radius",
    "Covariance",
    "Weight"
  ],
  "profiling_extra_cols_zero_filled": [
    "Duration",
    "Srate",
    "Drate",
    "Magnitue",
    "Radius",
    "Covariance",
    "Weight"
  ]
}
Profiling aligned: (389956, 49)
Attack-test aligned: (1614182, 49)


In [9]:
# وظيفة الخلية: حساب row hashes باستخدام schema الأصلي 39 فقط حتى يكون dedup/split مطابقاً للبروتوكول الأساسي قدر الإمكان.

def row_hashes(df, feature_cols):
    stable = df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(SENTINEL)
    return pd.util.hash_pandas_object(stable, index=False).astype(str)

def duplicate_summary(df, name):
    total = len(df)
    unique = int(df["__row_hash"].nunique()) if total else 0
    return {
        "pool": name,
        "rows": int(total),
        "unique_rows": int(unique),
        "duplicate_rows": int(total - unique),
        "duplicate_ratio": float((total - unique) / total) if total else 0.0,
        "hash_schema": "base_schema_39",
    }

profiling_aligned["__row_hash"] = row_hashes(profiling_aligned, BASE_SCHEMA_39)
attack_test_aligned["__row_hash"] = row_hashes(attack_test_aligned, BASE_SCHEMA_39)

duplicate_audit = pd.DataFrame([
    duplicate_summary(profiling_aligned, "profiling"),
    duplicate_summary(attack_test_aligned, "attack_test"),
])
duplicate_audit.to_csv(os.path.join(AUDITS_DIR, "schema_retention_duplicate_audit_base39_hash.csv"), index=False)

print(duplicate_audit.to_string(index=False))


       pool    rows  unique_rows  duplicate_rows  duplicate_ratio    hash_schema
  profiling  389956       360962           28994         0.074352 base_schema_39
attack_test 1614182      1612111            2071         0.001283 base_schema_39


In [10]:
# وظيفة الخلية: تطبيق dedup_on، إزالة أي تداخل attack/profiling، وتقسيم profiling إلى train/validation/test بنفس منطق البروتوكول الأصلي.

def overlap_count(df_a, df_b):
    return int(len(set(df_a["__row_hash"]).intersection(set(df_b["__row_hash"]))))

def assert_disjoint(df_a, df_b, name_a, name_b):
    ov = overlap_count(df_a, df_b)
    assert ov == 0, f"Overlap between {name_a} and {name_b}: {ov}"

def split_by_hash(df, seed):
    unique_hashes = df["__row_hash"].drop_duplicates().tolist()
    train_h, temp_h = train_test_split(
        unique_hashes,
        test_size=(VALID_RATIO + TEST_RATIO),
        random_state=seed,
        shuffle=True,
    )
    valid_h, test_h = train_test_split(
        temp_h,
        test_size=(TEST_RATIO / (VALID_RATIO + TEST_RATIO)),
        random_state=seed,
        shuffle=True,
    )
    train_df = df[df["__row_hash"].isin(train_h)].copy().reset_index(drop=True)
    valid_df = df[df["__row_hash"].isin(valid_h)].copy().reset_index(drop=True)
    test_df = df[df["__row_hash"].isin(test_h)].copy().reset_index(drop=True)
    return train_df, valid_df, test_df

profiling_work = profiling_aligned.drop_duplicates("__row_hash").copy().reset_index(drop=True)
attack_test_work = attack_test_aligned.drop_duplicates("__row_hash").copy().reset_index(drop=True)

profiling_hashes = set(profiling_work["__row_hash"])
attack_test_clean = attack_test_work[~attack_test_work["__row_hash"].isin(profiling_hashes)].copy().reset_index(drop=True)

normal_train, normal_validation, normal_test = split_by_hash(profiling_work, RANDOM_STATE)

assert_disjoint(normal_train, normal_validation, "normal_train", "normal_validation")
assert_disjoint(normal_train, normal_test, "normal_train", "normal_test")
assert_disjoint(normal_validation, normal_test, "normal_validation", "normal_test")
assert overlap_count(normal_train, attack_test_clean) == 0
assert overlap_count(normal_validation, attack_test_clean) == 0
assert overlap_count(normal_test, attack_test_clean) == 0

normal_part = normal_test.copy()
normal_part["final_label"] = 0
attack_part = attack_test_clean.copy()
attack_part["final_label"] = 1
final_test = pd.concat([normal_part, attack_part], ignore_index=True)

split_summary = {
    "setting": "retained_schema_stress_test_dedup_on_ocsvm",
    "hash_schema": "base_schema_39",
    "model_feature_count": len(RETAINED_SCHEMA_46),
    "profiling_rows_after_dedup": int(len(profiling_work)),
    "attack_test_rows_after_dedup": int(len(attack_test_work)),
    "attack_test_clean_rows": int(len(attack_test_clean)),
    "normal_train_rows": int(len(normal_train)),
    "normal_validation_rows": int(len(normal_validation)),
    "normal_test_rows": int(len(normal_test)),
    "final_test_rows": int(len(final_test)),
    "positive_class_prevalence": float(final_test["final_label"].mean()),
    "benign_test_rows_in_attack_test_clean": int(attack_test_clean["__source_file"].eq("Benign_test.pcap.csv").sum()),
}
pd.DataFrame([split_summary]).to_csv(os.path.join(SUMMARIES_DIR, "schema_retention_split_summary.csv"), index=False)

print(json.dumps(split_summary, indent=2, ensure_ascii=False))


/tmp/ipykernel_2900/1953410824.py:48: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_test = pd.concat([normal_part, attack_part], ignore_index=True)


{
  "setting": "retained_schema_stress_test_dedup_on_ocsvm",
  "hash_schema": "base_schema_39",
  "model_feature_count": 46,
  "profiling_rows_after_dedup": 360962,
  "attack_test_rows_after_dedup": 1612111,
  "attack_test_clean_rows": 1612111,
  "normal_train_rows": 216577,
  "normal_validation_rows": 72192,
  "normal_test_rows": 72193,
  "final_test_rows": 1684304,
  "positive_class_prevalence": 0.9571377851029268,
  "benign_test_rows_in_attack_test_clean": 37607
}


In [11]:
# وظيفة الخلية: تدريب OCSVM مرة واحدة على retained schema وحساب النتائج الرئيسية.

def safe_div(num, den):
    return float(num) / float(den) if den else np.nan

def confusion_to_metrics(tn, fp, fn, tp):
    tn, fp, fn, tp = map(int, [tn, fp, fn, tp])
    total = tn + fp + fn + tp
    accuracy = safe_div(tn + tp, total)
    precision = safe_div(tp, tp + fp)
    recall = safe_div(tp, tp + fn)
    specificity = safe_div(tn, tn + fp)
    fpr = safe_div(fp, tn + fp)
    prevalence = safe_div(tp + fn, total)
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else np.nan
    balanced_accuracy = (recall + specificity) / 2 if np.isfinite(recall) and np.isfinite(specificity) else np.nan
    denom = float(tp + fp) * float(tp + fn) * float(tn + fp) * float(tn + fn)
    mcc = ((float(tp) * float(tn)) - (float(fp) * float(fn))) / math.sqrt(denom) if denom > 0 else np.nan
    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "mcc": mcc,
        "specificity": specificity,
        "fpr": fpr,
        "balanced_accuracy": balanced_accuracy,
        "pr_auc_baseline": prevalence,
    }

def evaluate_scores(y_true, scores, threshold):
    y_true = np.asarray(y_true, dtype=int)
    scores = np.asarray(scores, dtype=float)
    preds = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, preds, labels=[0, 1]).ravel()
    metrics = {
        "roc_auc": float(roc_auc_score(y_true, scores)) if len(np.unique(y_true)) == 2 else np.nan,
        "pr_auc": float(average_precision_score(y_true, scores)) if len(np.unique(y_true)) == 2 else np.nan,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "threshold": float(threshold),
    }
    metrics.update(confusion_to_metrics(tn, fp, fn, tp))
    return metrics, preds

def transform_matrix(df, feature_cols, imputer=None, scaler=None, fit=False):
    X = df[feature_cols].replace([np.inf, -np.inf], np.nan)
    if fit:
        imputer = SimpleImputer(strategy="median")
        scaler = StandardScaler()
        X_imp = imputer.fit_transform(X)
        X_scaled = scaler.fit_transform(X_imp)
        return X_scaled, imputer, scaler
    X_imp = imputer.transform(X)
    X_scaled = scaler.transform(X_imp)
    return X_scaled

print("Fitting preprocessing on normal_train only...")
X_train, imputer, scaler = transform_matrix(normal_train, RETAINED_SCHEMA_46, fit=True)
X_val = transform_matrix(normal_validation, RETAINED_SCHEMA_46, imputer=imputer, scaler=scaler)
X_test = transform_matrix(final_test, RETAINED_SCHEMA_46, imputer=imputer, scaler=scaler)
y_test = final_test["final_label"].astype(int).to_numpy()

print("Training OCSVM retained-schema stress test...")
model = OneClassSVM(kernel="rbf", gamma="scale", nu=0.05)

fit_start = time.time()
model.fit(X_train)
fit_seconds = time.time() - fit_start

val_scores = -model.decision_function(X_val)
threshold = float(np.percentile(val_scores, 95))
test_scores = -model.decision_function(X_test)

metrics, preds = evaluate_scores(y_test, test_scores, threshold)

result_row = {
    "setting": "retained_schema_stress_test",
    "model": "ocsvm",
    "dedup": "dedup_on",
    "identity_hash_schema": "base_schema_39",
    "model_feature_schema": "base_39_plus_7_retained_attack_side_only",
    "feature_count": int(len(RETAINED_SCHEMA_46)),
    "train_rows_used": int(len(normal_train)),
    "validation_rows_used": int(len(normal_validation)),
    "test_rows_used": int(len(final_test)),
    "fit_time_seconds": float(fit_seconds),
    "threshold_percentile": 95,
    **metrics,
}

result_df = pd.DataFrame([result_row])
result_path = os.path.join(PUBLICATION_DIR, "schema_retention_ablation_ocsvm.csv")
result_df.to_csv(result_path, index=False)

print("Saved:", result_path)
print(result_df.to_string(index=False))


Fitting preprocessing on normal_train only...
Training OCSVM retained-schema stress test...
Saved: /content/schema_retention_ablation_20260507_005607/publication_checks/schema_retention_ablation_ocsvm.csv
                    setting model    dedup identity_hash_schema                     model_feature_schema  feature_count  train_rows_used  validation_rows_used  test_rows_used  fit_time_seconds  threshold_percentile  roc_auc  pr_auc    tn   fp  fn      tp  threshold  accuracy  precision  recall       f1      mcc  specificity      fpr  balanced_accuracy  pr_auc_baseline
retained_schema_stress_test ocsvm dedup_on       base_schema_39 base_39_plus_7_retained_attack_side_only             46           216577                 72192         1684304        730.094593                    95      1.0     1.0 68601 3592   0 1612111   0.013556  0.997867   0.997777     1.0 0.998887 0.973721     0.950244 0.049756           0.975122         0.957138


In [12]:
# وظيفة الخلية: حساب attack-only excluding Benign_test وper-family recall لتسهيل تفسير نتيجة ablation.

def infer_attack_family_from_file(filename):
    name = str(filename).lower()
    if "benign" in name:
        return "Benign_test"
    if "arp" in name:
        return "ARP spoofing"
    if "connect" in name and "mqtt" in name:
        return "MQTT connect flood"
    if "publish" in name and "mqtt" in name:
        return "MQTT publish flood"
    if "malformed" in name and "mqtt" in name:
        return "MQTT malformed data"
    if any(x in name for x in ["ping", "scan", "recon", "os_scan", "port_scan", "vul"]):
        return "Reconnaissance"
    if "ddos" in name:
        return "TCP/IP DDoS"
    if "dos" in name:
        return "TCP/IP DoS"
    return "Other"

prediction_df = final_test[META_COLS + ["final_label"]].copy()
prediction_df["anomaly_score"] = test_scores
prediction_df["prediction"] = preds
prediction_df["model"] = "ocsvm"
prediction_df["setting"] = "retained_schema_stress_test"

# Attack-only excluding Benign_test.
attack_only = prediction_df[prediction_df["__source_file"] != "Benign_test.pcap.csv"].copy()
attack_only_metrics, _ = evaluate_scores(
    attack_only["final_label"].astype(int).to_numpy(),
    attack_only["anomaly_score"].to_numpy(),
    threshold,
)
attack_only_row = {
    "setting": "retained_schema_stress_test",
    "model": "ocsvm",
    "treatment": "attack_only_excluding_benign",
    **attack_only_metrics,
}
attack_only_df = pd.DataFrame([attack_only_row])
attack_only_path = os.path.join(PUBLICATION_DIR, "schema_retention_attack_only_ocsvm.csv")
attack_only_df.to_csv(attack_only_path, index=False)

# Per-family recall on attack-only rows.
fam_rows = []
attack_rows = attack_only[attack_only["final_label"].astype(int).eq(1)].copy()
attack_rows["attack_family"] = attack_rows["__source_file"].map(infer_attack_family_from_file)

for fam, g in attack_rows.groupby("attack_family", sort=True):
    y = g["final_label"].astype(int).to_numpy()
    s = g["anomaly_score"].to_numpy()
    m, p = evaluate_scores(y, s, threshold)
    fam_rows.append({
        "setting": "retained_schema_stress_test",
        "model": "ocsvm",
        "attack_family": fam,
        "rows": int(len(g)),
        "tp": int(m["tp"]),
        "fn": int(m["fn"]),
        "recall": float(m["recall"]),
    })

family_df = pd.DataFrame(fam_rows)
family_path = os.path.join(PUBLICATION_DIR, "schema_retention_per_attack_family_ocsvm.csv")
family_df.to_csv(family_path, index=False)

print("Attack-only:")
print(attack_only_df.to_string(index=False))
print("\\nPer-family:")
print(family_df.to_string(index=False))


Attack-only:
                    setting model                    treatment  roc_auc  pr_auc    tn   fp  fn      tp  threshold  accuracy  precision  recall       f1      mcc  specificity      fpr  balanced_accuracy  pr_auc_baseline
retained_schema_stress_test ocsvm attack_only_excluding_benign      1.0     1.0 68601 3592   0 1574504   0.013556  0.997819   0.997724     1.0 0.998861 0.973695     0.950244 0.049756           0.975122         0.956159
\nPer-family:
                    setting model       attack_family    rows      tp  fn  recall
retained_schema_stress_test ocsvm        ARP spoofing    1744    1744   0     1.0
retained_schema_stress_test ocsvm  MQTT connect flood   45047   45047   0     1.0
retained_schema_stress_test ocsvm MQTT malformed data    1747    1747   0     1.0
retained_schema_stress_test ocsvm  MQTT publish flood   16921   16921   0     1.0
retained_schema_stress_test ocsvm      Reconnaissance   25607   25607   0     1.0
retained_schema_stress_test ocsvm         T

In [13]:
# وظيفة الخلية: محاولة تحميل صف baseline الأصلي من نتائج التشغيل النهائي السابقة للمقارنة في جدول واحد.

def is_results_root(path):
    path = Path(path)
    return (path / "metrics" / "metrics_main.csv").exists()

def zip_contains_metrics(zip_path):
    try:
        with zipfile.ZipFile(zip_path, "r") as z:
            return "metrics/metrics_main.csv" in set(z.namelist())
    except Exception:
        return False

def find_baseline_results():
    if MANUAL_BASELINE_RESULTS_ROOT and is_results_root(MANUAL_BASELINE_RESULTS_ROOT):
        return {"mode": "folder", "path": MANUAL_BASELINE_RESULTS_ROOT}
    if MANUAL_BASELINE_RESULTS_ZIP and zip_contains_metrics(MANUAL_BASELINE_RESULTS_ZIP):
        return {"mode": "zip", "path": MANUAL_BASELINE_RESULTS_ZIP}

    # بحث محدود عن publication_full*.zip أو metrics_main.csv.
    for root, dirs, files in os.walk(DRIVE_ROOT):
        if depth_from_root(root, DRIVE_ROOT) > 6:
            dirs[:] = []
            continue
        if "metrics_main.csv" in files and os.path.basename(root) == "metrics":
            result_root = os.path.dirname(root)
            if is_results_root(result_root):
                return {"mode": "folder", "path": result_root}
        for f in files:
            if f.lower().endswith(".zip") and "publication_full" in f.lower():
                zp = os.path.join(root, f)
                if zip_contains_metrics(zp):
                    return {"mode": "zip", "path": zp}
    return None

def load_baseline_metrics(source):
    if source is None:
        return pd.DataFrame()
    if source["mode"] == "folder":
        return pd.read_csv(os.path.join(source["path"], "metrics", "metrics_main.csv"))
    tmp = os.path.join(LOCAL_OUTPUT_ROOT, "baseline_extract")
    os.makedirs(tmp, exist_ok=True)
    with zipfile.ZipFile(source["path"], "r") as z:
        z.extract("metrics/metrics_main.csv", tmp)
    return pd.read_csv(os.path.join(tmp, "metrics", "metrics_main.csv"))

baseline_source = find_baseline_results()
print("Baseline source:", baseline_source)

comparison_rows = []

baseline_df = load_baseline_metrics(baseline_source)
if not baseline_df.empty:
    mask = (baseline_df["setting"].eq("dedup_on")) & (baseline_df["model"].eq("ocsvm"))
    if mask.any():
        b = baseline_df.loc[mask].iloc[0].to_dict()
        comparison_rows.append({
            "condition": "profiling_only_schema_baseline",
            "feature_count": 39,
            "notes": "Main paper baseline; attack-side-only columns removed.",
            "roc_auc": b.get("roc_auc"),
            "pr_auc": b.get("pr_auc"),
            "f1": b.get("f1"),
            "mcc": b.get("mcc"),
            "fpr": b.get("fpr"),
            "recall": b.get("recall"),
            "balanced_accuracy": b.get("balanced_accuracy"),
            "tn": b.get("tn"),
            "fp": b.get("fp"),
            "fn": b.get("fn"),
            "tp": b.get("tp"),
            "fit_time_seconds": b.get("fit_time_seconds"),
        })

r = result_row
comparison_rows.append({
    "condition": "retained_schema_stress_test",
    "feature_count": len(RETAINED_SCHEMA_46),
    "notes": "Adds seven attack-side-only columns; profiling values zero-filled.",
    "roc_auc": r.get("roc_auc"),
    "pr_auc": r.get("pr_auc"),
    "f1": r.get("f1"),
    "mcc": r.get("mcc"),
    "fpr": r.get("fpr"),
    "recall": r.get("recall"),
    "balanced_accuracy": r.get("balanced_accuracy"),
    "tn": r.get("tn"),
    "fp": r.get("fp"),
    "fn": r.get("fn"),
    "tp": r.get("tp"),
    "fit_time_seconds": r.get("fit_time_seconds"),
})

comparison_df = pd.DataFrame(comparison_rows)
comparison_path = os.path.join(PUBLICATION_DIR, "schema_retention_ablation_comparison_for_manuscript.csv")
comparison_df.to_csv(comparison_path, index=False)

print("Saved:", comparison_path)
print(comparison_df.to_string(index=False))


Baseline source: {'mode': 'zip', 'path': '/content/drive/MyDrive/IoMT_Project/Experiments/CICIoMT2024_NormalOnly_DuplicateAware_publication_full/publication_full_20260506_013718.zip'}
Saved: /content/schema_retention_ablation_20260507_005607/publication_checks/schema_retention_ablation_comparison_for_manuscript.csv
                     condition  feature_count                                                              notes  roc_auc  pr_auc       f1      mcc      fpr  recall  balanced_accuracy    tn   fp  fn      tp  fit_time_seconds
profiling_only_schema_baseline             39             Main paper baseline; attack-side-only columns removed.      1.0     1.0 0.998887 0.973721 0.049756     1.0           0.975122 68601 3592   0 1612111        609.920043
   retained_schema_stress_test             46 Adds seven attack-side-only columns; profiling values zero-filled.      1.0     1.0 0.998887 0.973721 0.049756     1.0           0.975122 68601 3592   0 1612111        730.094593


In [14]:
# وظيفة الخلية: حفظ ملخص قابل للاقتباس في الورقة مع تفسير منهجي حذر للتجربة.

interpretation = {
    "recommended_section_title": "Schema-Retention Stress Test",
    "recommended_text": (
        "This ablation is not a deployable feature design. It intentionally retains the seven "
        "attack-side-only numeric columns and zero-fills them in profiling traffic to quantify "
        "how schema retention would affect the operating point if those columns were not removed."
    ),
    "main_output_files": [
        "publication_checks/schema_retention_ablation_ocsvm.csv",
        "publication_checks/schema_retention_attack_only_ocsvm.csv",
        "publication_checks/schema_retention_per_attack_family_ocsvm.csv",
        "publication_checks/schema_retention_ablation_comparison_for_manuscript.csv",
    ],
}
save_json(interpretation, os.path.join(PUBLICATION_DIR, "schema_retention_interpretation_note.json"))

print(json.dumps(interpretation, ensure_ascii=False, indent=2))


{
  "recommended_section_title": "Schema-Retention Stress Test",
  "recommended_text": "This ablation is not a deployable feature design. It intentionally retains the seven attack-side-only numeric columns and zero-fills them in profiling traffic to quantify how schema retention would affect the operating point if those columns were not removed.",
  "main_output_files": [
    "publication_checks/schema_retention_ablation_ocsvm.csv",
    "publication_checks/schema_retention_attack_only_ocsvm.csv",
    "publication_checks/schema_retention_per_attack_family_ocsvm.csv",
    "publication_checks/schema_retention_ablation_comparison_for_manuscript.csv"
  ]
}


In [15]:
# وظيفة الخلية: إنشاء manifest وضغط مخرجات ablation ونسخها إلى Google Drive.

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

manifest_rows = []
for root, _, files in os.walk(LOCAL_OUTPUT_ROOT):
    for name in files:
        if name.endswith((".csv", ".json")):
            full = os.path.join(root, name)
            manifest_rows.append({
                "relative_path": os.path.relpath(full, LOCAL_OUTPUT_ROOT),
                "size_bytes": os.path.getsize(full),
                "sha256": sha256_file(full),
            })

manifest_df = pd.DataFrame(manifest_rows).sort_values("relative_path")
manifest_path = os.path.join(PUBLICATION_DIR, "schema_retention_manifest_sha256.csv")
manifest_df.to_csv(manifest_path, index=False)

os.makedirs(DRIVE_OUTPUT_PARENT, exist_ok=True)
local_zip_base = os.path.join("/content", RUN_TAG)
local_zip_path = shutil.make_archive(local_zip_base, "zip", LOCAL_OUTPUT_ROOT)
drive_zip_path = os.path.join(DRIVE_OUTPUT_PARENT, os.path.basename(local_zip_path))
shutil.copy2(local_zip_path, drive_zip_path)

print("Local output root:", LOCAL_OUTPUT_ROOT)
print("Drive output parent:", DRIVE_OUTPUT_PARENT)
print("Drive ZIP:", drive_zip_path)

print("\\nSend these files:")
needed = [
    os.path.join(PUBLICATION_DIR, "schema_retention_ablation_ocsvm.csv"),
    os.path.join(PUBLICATION_DIR, "schema_retention_attack_only_ocsvm.csv"),
    os.path.join(PUBLICATION_DIR, "schema_retention_per_attack_family_ocsvm.csv"),
    os.path.join(PUBLICATION_DIR, "schema_retention_ablation_comparison_for_manuscript.csv"),
    os.path.join(PUBLICATION_DIR, "schema_retention_interpretation_note.json"),
    os.path.join(PUBLICATION_DIR, "schema_retention_manifest_sha256.csv"),
    drive_zip_path,
]
for p in needed:
    print("-", p)


Local output root: /content/schema_retention_ablation_20260507_005607
Drive output parent: /content/drive/MyDrive/schema_retention_ablation_results
Drive ZIP: /content/drive/MyDrive/schema_retention_ablation_results/schema_retention_ablation_20260507_005607.zip
\nSend these files:
- /content/schema_retention_ablation_20260507_005607/publication_checks/schema_retention_ablation_ocsvm.csv
- /content/schema_retention_ablation_20260507_005607/publication_checks/schema_retention_attack_only_ocsvm.csv
- /content/schema_retention_ablation_20260507_005607/publication_checks/schema_retention_per_attack_family_ocsvm.csv
- /content/schema_retention_ablation_20260507_005607/publication_checks/schema_retention_ablation_comparison_for_manuscript.csv
- /content/schema_retention_ablation_20260507_005607/publication_checks/schema_retention_interpretation_note.json
- /content/schema_retention_ablation_20260507_005607/publication_checks/schema_retention_manifest_sha256.csv
- /content/drive/MyDrive/schema

## الملفات المطلوبة بعد التشغيل

أرسل لي هذه الملفات بعد انتهاء التشغيل:

```text
publication_checks/schema_retention_ablation_ocsvm.csv
publication_checks/schema_retention_attack_only_ocsvm.csv
publication_checks/schema_retention_per_attack_family_ocsvm.csv
publication_checks/schema_retention_ablation_comparison_for_manuscript.csv
publication_checks/schema_retention_interpretation_note.json
publication_checks/schema_retention_manifest_sha256.csv
```

وأرسل أيضاً ملف ZIP الناتج من Google Drive:

```text
schema_retention_ablation_results/schema_retention_ablation_*.zip
```
